# Kimi K2.6 Agentic Workflow

**How to run:**
1. Add `TOGETHER_API_KEY` and `TAVILY_API_KEY` to your Kaggle secrets and enable notebook access.
2. Upload your `your_chats.json` to the notebook working directory (or a Kaggle dataset).
3. Run cells top to bottom.

In [ ]:
# Phase 1 — Install dependencies
%pip install -q requests tenacity pydantic

In [ ]:
# Phase 1 — Load secrets
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
TOGETHER_API_KEY = _secrets.get_secret("TOGETHER_API_KEY")
TAVILY_API_KEY   = _secrets.get_secret("TAVILY_API_KEY")

KIMI_BASE_URL = "https://api.together.ai/v1/chat/completions"
KIMI_MODEL    = "moonshotai/Kimi-K2.6"
TAVILY_URL    = "https://api.tavily.com/search"

In [ ]:
# Phase 2 — kimi_call
import requests
import tenacity

@tenacity.retry(
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=8),
    stop=tenacity.stop_after_attempt(3),
    retry=tenacity.retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True,
)
def kimi_call(messages, max_tokens=512):
    response = requests.post(
        KIMI_BASE_URL,
        headers={
            "Authorization": f"Bearer {TOGETHER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": KIMI_MODEL,
            "messages": messages,
            "temperature": 0.6,
            "max_tokens": max_tokens,
        },
        timeout=30,
    )
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    if not content:
        raise ValueError(f"Empty content from API. Full response: {response.json()}")
    return content

In [ ]:
# Phase 2 — web_search
_search_cache = {}

@tenacity.retry(
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=8),
    stop=tenacity.stop_after_attempt(3),
    retry=tenacity.retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True,
)
def web_search(query, max_results=5):
    if query in _search_cache:
        return _search_cache[query]
    response = requests.post(
        TAVILY_URL,
        json={
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": max_results,
            "search_depth": "basic",
            "include_answer": False,
            "after_date": "2022-01-01",
        },
        timeout=10,
    )
    response.raise_for_status()
    results = response.json()
    _search_cache[query] = results
    return results

In [ ]:
# Phase 2 — count_tokens
def count_tokens(text):
    return int(len(text.split()) * 1.3)

In [ ]:
# Phase 2 — Smoke tests
# kimi_call
reply = kimi_call([{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=50)
print("kimi_call :", reply)

# web_search
results = web_search("latest AI news", max_results=1)
print("web_search keys:", list(results.keys()))
print("first result  :", results["results"][0]["title"])

# count_tokens
print("count_tokens  :", count_tokens("hello world this is a test sentence"))

---
## Phase 3 — Style Extraction from Chat History

In [ ]:
# Phase 3 — Extract style prompt from chat history
import json, zipfile, glob, os

# Locate the Claude export zip (works for Kaggle dataset upload or notebook file upload)
_candidates = (
    glob.glob("/kaggle/input/**/*.zip", recursive=True) +
    glob.glob("./*.zip") +
    glob.glob("/kaggle/working/*.zip")
)
if not _candidates:
    raise FileNotFoundError("No zip file found. Upload your Claude export zip to the notebook or a Kaggle dataset.")
_zip_path = _candidates[0]
print(f"Using: {_zip_path}")

# Read conversations.json directly from the zip (no disk extraction needed)
with zipfile.ZipFile(_zip_path) as zf:
    with zf.open("conversations.json") as f:
        conversations = json.load(f)

# Collect all substantive human messages (> 150 chars to filter out one-liners)
human_messages = []
for convo in conversations:
    for msg in convo.get("chat_messages", []):
        if msg.get("sender") == "human":
            text = msg.get("text", "").strip()
            if len(text) > 150:
                human_messages.append(text)

# Sort longest-first (most thoughtful messages tend to be longer), take top 15
human_messages.sort(key=len, reverse=True)
examples = human_messages[:15]

# Trim each example to 400 chars so the system prompt stays under ~8k tokens
examples = [m[:400] + ("…" if len(m) > 400 else "") for m in examples]

YOUR_STYLE_PROMPT = (
    "You are an AI assistant that thinks and communicates in the style of a specific person, "
    "based on their actual messages. Mirror their vocabulary, reasoning depth, structure, and tone. "
    "Do not mention that you are mimicking anyone — just respond naturally in their style.\n\n"
    "Here are representative examples of how this person writes and thinks:\n\n"
    + "\n\n---\n\n".join(f"Example {i+1}:\n{ex}" for i, ex in enumerate(examples))
)

print(f"Loaded {len(conversations)} conversations, {len(human_messages)} substantive human messages.")
print(f"Style prompt built from top {len(examples)} examples.")
print(f"Prompt token estimate: {count_tokens(YOUR_STYLE_PROMPT)}")

In [ ]:
# Phase 3 — Validate style prompt on 2 quick test questions
_validation_questions = [
    "What's your take on using prediction markets to inform investment decisions?",
    "How would you approach analysing a company before an earnings call?",
]

for q in _validation_questions:
    print(f"Q: {q}")
    reply = kimi_call(
        [{"role": "system", "content": YOUR_STYLE_PROMPT},
         {"role": "user",   "content": q}],
        max_tokens=300,
    )
    print(f"A: {reply}")
    print("─" * 60)